# Giving the transformer a sense of recency and of speakers

**Research question.** Under what conditions can a transformer outperform a recurrent network on this task? In `fisher_next_turn_prediction.ipynb`, the tuned transformer beat the GRU but not the GRU with attention. This notebook tests changes that might close that gap: a bias toward recent turns and a bias that knows which past turns came from the same speaker, then (sections 9 and 10) telling the models who speaks next and adding regularization.

**Why it might help.** The transformer was the best model when the speaker changed, but the worst when the same speaker kept talking. A GRU's state is dominated by the most recent turns, which suits that case. The transformer adds absolute (sinusoidal) positions to its input, so it has to *learn* that recent turns matter more. **ALiBi** (Press, Smith & Lewis, 2022, *Train Short, Test Long*) builds recency into attention directly. A **speaker bias** lets each attention head prefer, or avoid, the earlier turns of the person who spoke last.

Follows up on Prof. Matthew McNeill's question: can a transformer outperform a recurrent network at predicting upcoming speech features?

## What is compared

Speech features only (the main experiment of the first notebook). Each model uses the configuration that won the tuning sweep there.

| model | positions | speaker bias |
|---|---|---|
| GRU (hidden 256) | – | – |
| GRU + attention (hidden 128) | – | – |
| transformer (4 layers, size 256) | sinusoidal, added to the input | no |
| transformer + ALiBi | ALiBi: attention to a turn d turns back is lowered by m·d | no |
| transformer + speaker bias | sinusoidal | yes |
| transformer + ALiBi + speaker bias | ALiBi | yes |

## Same setup as the main notebook, with one change

- Same data checks, split by `conv_id` (seed 0), standardization from the training set, start turn, loss on measured values only, Adam with gradient clipping, learning-rate warmup over 500 steps, early stopping on validation loss (patience 3, at most 100 epochs), bf16 autocast on the GPU.
- **Seeds.** The first notebook trained each model once, and retraining moved test MSE by up to 0.001, about half the gap we are trying to close. Here every model is trained with **3 seeds**. The learning rate (1e-3 or 3e-4) is chosen on validation with seed 0, then seeds 1 and 2 use that learning rate. The seed sets the initial weights, dropout, and the order of training batches.
- **Resumable.** Each run saves its result and weights as soon as it finishes, so an interrupted run picks up where it stopped.

## 1. Setup

In [1]:
import json
import math
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from fisher_data import (FEATURES, N_FEAT, ConversationDataset, build_conversations, collate_conversations,
                         load_turns, split_conversations)
from imdb_data import BucketBatchSampler

SMOKE_TEST = os.environ.get("FISHER_SMOKE") == "1"  # small run used to check the notebook end to end
SPLIT_SEED = 0
SEEDS = [0, 1, 2]
LEARNING_RATES = [1e-3, 3e-4]
WARMUP_STEPS = 500
MAX_EPOCHS = 2 if SMOKE_TEST else 100
PATIENCE = 3
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = device.type == "cuda" and torch.cuda.is_bf16_supported()
# Smoke runs write to the git-ignored checkpoints folder, so they never mix with real results.
RESULTS_DIR = Path("checkpoints/attention_bias_smoke" if SMOKE_TEST else "results/attention_bias")
CHECKPOINT_DIR = Path("checkpoints/attention_bias_smoke" if SMOKE_TEST else "checkpoints/attention_bias")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "",
      "| bf16:", USE_BF16, "| torch", torch.__version__, "| smoke test:", SMOKE_TEST)

device: cuda NVIDIA GeForce RTX 4050 Laptop GPU | bf16: True | torch 2.11.0+cu128 | smoke test: False


## 2. Data, checked as in the main notebook

Only summary numbers are printed: the transcripts are licensed and are not used here.

In [2]:
turns = load_turns("data/fisher/fisher_data_simple.csv")
by_conv = turns.groupby("conv_id")["turn_id"]
turn_counts = by_conv.size()
assert (by_conv.min() == 1).all(), "every conversation starts at turn 1"
assert ((by_conv.max() - by_conv.min() + 1) == turn_counts).all(), "no gaps in turn_id"
assert not turns.duplicated(["conv_id", "turn_id"]).any(), "no duplicate turns"
assert set(turns.side) == {"A", "B"}
print(f"{len(turns):,} turns in {turns.conv_id.nunique():,} conversations")

train_ids, val_ids, test_ids = split_conversations(turns.conv_id.unique(), seed=SPLIT_SEED)
if SMOKE_TEST:
    train_ids, val_ids, test_ids = (set(sorted(ids)[:n]) for ids, n in [(train_ids, 400), (val_ids, 100), (test_ids, 100)])
assert not (train_ids & val_ids) and not (train_ids & test_ids) and not (val_ids & test_ids)
train_turns = turns[turns.conv_id.isin(train_ids)]
feature_mean = train_turns[FEATURES].mean().to_numpy(np.float32)
feature_std = train_turns[FEATURES].std().to_numpy(np.float32)

# Words are not used in this notebook, so the vocabulary is empty.
splits = {name: build_conversations(turns, ids, feature_mean, feature_std, vocab={})
          for name, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]}
conv = splits["train"][0]
assert torch.equal(conv.inputs[1:, :N_FEAT], conv.targets[:-1])
print(f"conversations: train {len(train_ids):,} | val {len(val_ids):,} | test {len(test_ids):,}")
print("turns per split:", {k: sum(len(c) for c in v) for k, v in splits.items()})

2,076,692 turns in 10,998 conversations


conversations: train 8,798 | val 1,099 | test 1,101
turns per split: {'train': 1657699, 'val': 208068, 'test': 210925}


In [3]:
def make_loader(name, shuffle, seed=0):
    conversations = splits[name]
    sampler = BucketBatchSampler([len(c) for c in conversations], BATCH_SIZE, shuffle=shuffle,
                                 drop_last=False, seed=seed)
    return DataLoader(ConversationDataset(conversations), batch_sampler=sampler, collate_fn=collate_conversations)


# Validation and test loaders are fixed. Each training run gets its own loader, seeded by the run's seed,
# so a model's batch order doesn't depend on which models were trained before it.
loaders = {"val": make_loader("val", False), "test": make_loader("test", False)}


def to_device(batch):
    return {k: v.to(device) for k, v in batch.items()}

## 3. Models

The GRU models and the turn encoder are the same as in the main notebook (speech features only: the 7 features, 7 missing flags and the speaker go through one linear layer).

The transformer is written out here instead of using `nn.TransformerEncoderLayer`, because both changes add a term to the attention scores before the softmax:

$$\text{score}_{h}(t, i) = \frac{q_t \cdot k_i}{\sqrt{d}} \;-\; m_h\,(t - i) \;+\; b^{\ell}_{h,\text{same}}\,[\text{same speaker}] \;+\; b^{\ell}_{h,\text{other}}\,[\text{other speaker}]$$

- **Causal mask:** turns after t get a score of −∞, as before.
- **ALiBi** (the $-m_h(t-i)$ term): each of the 4 heads has a fixed slope $m_h \in \{1/4, 1/16, 1/64, 1/256\}$, so some heads look mostly at the last few turns and others far back. With ALiBi, no positions are added to the input.
- **Speaker bias:** for each layer ℓ and head h, two learned numbers, added when past turn i is by the same speaker as turn t (the most recent turn) or by the other speaker. They start at 0. The start turn has no speaker and gets neither.
- Everything else matches the first notebook's transformer: pre-LayerNorm blocks, 4 heads, feed-forward size twice the model size, ReLU, dropout 0.1, a final LayerNorm. The plain "transformer" row is re-run with this implementation, so the comparison only changes the bias terms.

In [4]:
def causal_mask(length, device):
    return torch.triu(torch.ones(length, length, dtype=torch.bool, device=device), diagonal=1)


class TurnEncoder(nn.Module):
    def __init__(self, model_dim=128):
        super().__init__()
        self.proj = nn.Linear(2 * N_FEAT + 2, model_dim)

    def forward(self, batch):
        return self.proj(batch["inputs"])  # 7 features, 7 missing flags, 2 speaker


class GRUModel(nn.Module):
    def __init__(self, encoder, hidden=128, attention=False, dropout=0.1):
        super().__init__()
        self.encoder, self.attention = encoder, attention
        self.gru = nn.GRU(encoder.proj.out_features, hidden, batch_first=True)
        if attention:
            self.W = nn.Linear(hidden, hidden, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(2 * hidden if attention else hidden, N_FEAT)

    def forward(self, batch):
        h, _ = self.gru(self.encoder(batch))
        if self.attention:
            scores = h @ self.W(h).transpose(1, 2) / math.sqrt(h.shape[-1])
            scores = scores.masked_fill(causal_mask(h.shape[1], h.device), -math.inf)
            h = torch.cat([h, torch.softmax(scores, dim=-1) @ h], dim=-1)
        return self.head(self.dropout(h))


class Block(nn.Module):
    # Pre-LayerNorm transformer block whose attention scores take an additive bias.
    def __init__(self, dim, heads, ff_dim, dropout):
        super().__init__()
        self.heads, self.head_dim = heads, dim // heads
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)
        self.ff = nn.Sequential(nn.Linear(dim, ff_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(ff_dim, dim))
        self.attn_dropout, self.dropout = nn.Dropout(dropout), nn.Dropout(dropout)

    def forward(self, x, bias):
        size, length, dim = x.shape
        q, k, v = self.qkv(self.norm1(x)).view(size, length, 3, self.heads, self.head_dim).permute(2, 0, 3, 1, 4)
        scores = (q @ k.transpose(-2, -1)).float() / math.sqrt(self.head_dim) + bias   # (batch, heads, t, i)
        weights = self.attn_dropout(torch.softmax(scores, dim=-1))
        attended = (weights.to(v.dtype) @ v).transpose(1, 2).reshape(size, length, dim)
        x = x + self.dropout(self.out(attended))
        return x + self.dropout(self.ff(self.norm2(x)))


class BiasedTransformer(nn.Module):
    def __init__(self, encoder, alibi, speaker_bias, layers=4, heads=4, dropout=0.1, max_len=1024):
        super().__init__()
        dim = encoder.proj.out_features
        self.encoder, self.alibi, self.heads = encoder, alibi, heads
        self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim, dropout) for _ in range(layers)])
        self.norm, self.dropout, self.head = nn.LayerNorm(dim), nn.Dropout(dropout), nn.Linear(dim, N_FEAT)
        self.register_buffer("slopes", torch.tensor([2.0 ** (-8 * (h + 1) / heads) for h in range(heads)]))
        self.speaker_bias = nn.Parameter(torch.zeros(layers, heads, 2)) if speaker_bias else None
        position = torch.arange(max_len).unsqueeze(1)
        freq = torch.exp(torch.arange(0, dim, 2) * (-math.log(10000.0) / dim))
        pe = torch.zeros(max_len, dim)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(position * freq), torch.cos(position * freq)
        self.register_buffer("positions", pe)

    def forward(self, batch):
        x = self.encoder(batch)
        size, length, _ = x.shape
        if not self.alibi:
            x = x + self.positions[:length]
        x = self.dropout(x)
        index = torch.arange(length, device=x.device)
        bias = torch.zeros(length, length, device=x.device).masked_fill(causal_mask(length, x.device), -math.inf)
        bias = bias.expand(1, self.heads, length, length)
        if self.alibi:
            distance = (index.unsqueeze(1) - index.unsqueeze(0)).clamp(min=0).float()     # t - i
            bias = bias - self.slopes.view(1, -1, 1, 1) * distance
        if self.speaker_bias is not None:
            speaker = batch["inputs"][..., -2:]                                           # one-hot A/B, zeros for the start turn
            same = speaker @ speaker.transpose(1, 2)                                      # 1 if turns t and i have the same speaker
            has = speaker.sum(-1)
            other = has.unsqueeze(2) * has.unsqueeze(1) - same                            # 1 if both have a speaker and it differs
            same, other = same.unsqueeze(1), other.unsqueeze(1)
        for layer, block in enumerate(self.blocks):
            layer_bias = bias
            if self.speaker_bias is not None:
                b = self.speaker_bias[layer]
                layer_bias = bias + b[:, 0].view(1, -1, 1, 1) * same + b[:, 1].view(1, -1, 1, 1) * other
            x = block(x, layer_bias)
        return self.head(self.norm(x))


MODELS = {
    "GRU": lambda: GRUModel(TurnEncoder(), hidden=256),
    "GRU + attention": lambda: GRUModel(TurnEncoder(), hidden=128, attention=True),
    "transformer": lambda: BiasedTransformer(TurnEncoder(256), alibi=False, speaker_bias=False),
    "transformer + ALiBi": lambda: BiasedTransformer(TurnEncoder(256), alibi=True, speaker_bias=False),
    "transformer + speaker bias": lambda: BiasedTransformer(TurnEncoder(256), alibi=False, speaker_bias=True),
    "transformer + ALiBi + speaker bias": lambda: BiasedTransformer(TurnEncoder(256), alibi=True, speaker_bias=True),
}
for name, build in MODELS.items():
    print(f"{name:36s} {sum(p.numel() for p in build().parameters()):>9,} parameters")

GRU                                    300,423 parameters
GRU + attention                        119,431 parameters
transformer                          2,115,079 parameters
transformer + ALiBi                  2,115,079 parameters


transformer + speaker bias           2,115,111 parameters
transformer + ALiBi + speaker bias   2,115,111 parameters


## 4. Sanity checks

1. **Causality**, as in the main notebook: change every input after turn k and check that the outputs up to turn k don't change. This also covers the speaker bias, which compares turn t with earlier turns only. It runs on the CPU, where results are exactly repeatable.
2. **The bias terms do what they say:** with ALiBi, a head's attention score drops by exactly its slope for each turn further back, and the speaker-bias matrices mark same-speaker and other-speaker pairs correctly.

In [5]:
def perturb_after(batch, k):
    changed = {key: value.clone() for key, value in batch.items()}
    changed["inputs"][:, k + 1:] = torch.randn_like(changed["inputs"][:, k + 1:])
    return changed


check_batch = next(iter(loaders["val"]))  # on the CPU
k = 20
for name, build in MODELS.items():
    torch.manual_seed(0)
    model = build().eval()
    if getattr(model, "speaker_bias", None) is not None:
        with torch.no_grad():
            model.speaker_bias.normal_()  # non-zero, so the check exercises the speaker terms
    with torch.no_grad():
        original, changed = model(check_batch), model(perturb_after(check_batch, k))
    assert torch.allclose(original[:, :k + 1], changed[:, :k + 1], atol=1e-5), name
    assert (original[:, k + 1:] - changed[:, k + 1:]).abs().max() > 1e-3, name
print(f"causality check passed for all {len(MODELS)} models")

# ALiBi: for a fixed query turn, scores with q = k = 0 equal -slope x distance.
model = MODELS["transformer + ALiBi"]().eval()
length = 6
index = torch.arange(length)
distance = (index.unsqueeze(1) - index.unsqueeze(0)).clamp(min=0).float()
alibi = -model.slopes.view(-1, 1, 1) * distance
assert torch.allclose(alibi[:, 5, 4] - alibi[:, 5, 3], model.slopes)

# Speaker matrices on a hand-made sequence: start turn, A, A, B
speaker = torch.tensor([[[0., 0.], [1., 0.], [1., 0.], [0., 1.]]])
same = speaker @ speaker.transpose(1, 2)
other = speaker.sum(-1).unsqueeze(2) * speaker.sum(-1).unsqueeze(1) - same
assert same[0, 2, 1] == 1 and other[0, 3, 1] == 1 and same[0, 3, 1] == 0 and same[0, 1, 0] == 0 and other[0, 1, 0] == 0
print("ALiBi slopes and speaker matrices check out")

causality check passed for all 6 models
ALiBi slopes and speaker matrices check out


## 5. Training

The training loop is the one from the main notebook, with the seed and the per-run training loader added. Each finished run writes a small JSON file (learning rate, seed, best epoch, validation MSE, history) and its weights. If the file already exists, the run is loaded instead of retrained.

In [6]:
def autocast():
    return torch.autocast(device.type, dtype=torch.bfloat16, enabled=USE_BF16)


def evaluate(model, loader):
    sse, count = torch.zeros(N_FEAT), torch.zeros(N_FEAT)
    with torch.no_grad():
        for batch in loader:
            batch = to_device(batch)
            with autocast():
                prediction = model(batch)
            error = (prediction.float() - batch["targets"]) ** 2 * batch["observed"]
            sse += error.sum(dim=(0, 1)).cpu()
            count += batch["observed"].sum(dim=(0, 1)).cpu()
    return float(sse.sum() / count.sum())


def train(name, lr, seed, weight_decay=0.0):
    # weight_decay > 0 switches to AdamW (decoupled weight decay); 0 is plain Adam, as in the main notebook
    torch.manual_seed(seed)
    model = MODELS[name]().to(device)
    optimizer = (torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay) if weight_decay
                 else torch.optim.Adam(model.parameters(), lr=lr))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: min(1.0, (step + 1) / WARMUP_STEPS))
    train_loader = make_loader("train", True, seed=seed)
    history, best = [], (math.inf, None, 0)
    for epoch in range(1, MAX_EPOCHS + 1):
        start = time.time()
        model.train()
        train_sse, train_count = 0.0, 0.0
        for batch in train_loader:
            batch = to_device(batch)
            with autocast():
                prediction = model(batch)
            squared = (prediction.float() - batch["targets"]) ** 2 * batch["observed"]
            loss = squared.sum() / batch["observed"].sum()
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            train_sse += squared.detach().sum().item()
            train_count += batch["observed"].sum().item()
        model.eval()
        val = evaluate(model, loaders["val"])
        history.append({"epoch": epoch, "train": train_sse / train_count, "val": val, "seconds": time.time() - start})
        if val < best[0]:
            best = (val, {k: v.detach().clone() for k, v in model.state_dict().items()}, epoch)
        elif epoch - best[2] >= PATIENCE:
            break
    model.load_state_dict(best[1])
    return model.eval(), history, best[2]


def slug(name, lr, seed, weight_decay=0.0):
    base = f"{name.replace(' + ', '_').replace(' ', '-').replace('(', '').replace(')', '')}_lr{lr:g}_seed{seed}"
    return base + (f"_wd{weight_decay:g}" if weight_decay else "")


def run(name, lr, seed, weight_decay=0.0):
    path = RESULTS_DIR / f"{slug(name, lr, seed, weight_decay)}.json"
    if path.exists():
        return json.loads(path.read_text())
    start = time.time()
    model, history, best_epoch = train(name, lr, seed, weight_decay)
    record = {"model": name, "lr": lr, "seed": seed, "weight decay": weight_decay, "best epoch": best_epoch, "val MSE": history[best_epoch - 1]["val"],
              "epochs run": len(history), "minutes": (time.time() - start) / 60, "history": history}
    torch.save(model.state_dict(), CHECKPOINT_DIR / f"{slug(name, lr, seed, weight_decay)}.pt")
    path.write_text(json.dumps(record, indent=1))
    print(f"  -> {name} | lr {lr:g} | wd {weight_decay:g} | seed {seed}: val MSE {record['val MSE']:.4f}, best epoch {best_epoch}, "
          f"{record['minutes']:.1f} min", flush=True)
    return record


def load_model(name, lr, seed, weight_decay=0.0):
    model = MODELS[name]().to(device)
    model.load_state_dict(torch.load(CHECKPOINT_DIR / f"{slug(name, lr, seed, weight_decay)}.pt", map_location=device))
    return model.eval()

## 6. Run every model: learning rate chosen with seed 0, then seeds 1 and 2

In [7]:
chosen_lr = {}
for name in MODELS:
    print(name, flush=True)
    first = {lr: run(name, lr, SEEDS[0]) for lr in LEARNING_RATES}
    chosen_lr[name] = min(first, key=lambda lr: first[lr]["val MSE"])
    for seed in SEEDS[1:]:
        run(name, chosen_lr[name], seed)
print("chosen learning rates:", chosen_lr)

GRU


GRU + attention


transformer


transformer + ALiBi


transformer + speaker bias


transformer + ALiBi + speaker bias


chosen learning rates: {'GRU': 0.0003, 'GRU + attention': 0.0003, 'transformer': 0.0003, 'transformer + ALiBi': 0.0003, 'transformer + speaker bias': 0.0003, 'transformer + ALiBi + speaker bias': 0.0003}


## 7. Results

Test MSE for each seed at the chosen learning rate, the mean and spread over the 3 seeds, and the difference from GRU + attention (negative = better than GRU + attention). The training mean scores about 1.02 on this test set.

In [8]:
def per_conversation_errors(model, loader, turn_type=None):
    sse, count = [], []
    with torch.no_grad():
        for batch in loader:
            batch = to_device(batch)
            observed = batch["observed"]
            if turn_type is not None:
                observed = observed & (batch["turn_type"] == turn_type).unsqueeze(-1)
            with autocast():
                prediction = model(batch)
            error = ((prediction.float() - batch["targets"]) ** 2) * observed
            sse.append(error.sum(dim=(1, 2)).cpu())
            count.append(observed.sum(dim=(1, 2)).cpu())
    return torch.cat(sse).numpy(), torch.cat(count).numpy()


errors = {}   # name -> per-seed (sse per conversation) for all turns, same-speaker turns, speaker-change turns
rows = []
for name in MODELS:
    lr = chosen_lr[name]
    per_seed = []
    for seed in SEEDS:
        model = load_model(name, lr, seed)
        per_seed.append({kind: per_conversation_errors(model, loaders["test"], turn_type)
                         for kind, turn_type in [("all", None), ("same speaker", 1), ("speaker change", 2)]})
    errors[name] = per_seed
    test = [e["all"][0].sum() / e["all"][1].sum() for e in per_seed]
    rows.append({"model": name, "lr": lr,
                 **{f"test MSE (seed {s})": t for s, t in zip(SEEDS, test)},
                 "mean": np.mean(test), "std": np.std(test, ddof=1),
                 "same speaker continues": np.mean([e["same speaker"][0].sum() / e["same speaker"][1].sum() for e in per_seed]),
                 "speaker change": np.mean([e["speaker change"][0].sum() / e["speaker change"][1].sum() for e in per_seed])})
results = pd.DataFrame(rows)
reference = results.set_index("model").loc["GRU + attention", "mean"]
results["vs GRU + attention"] = results["mean"] - reference
display(results.round(4))

,model,lr,test MSE (seed 0),test MSE (seed 1),test MSE (seed 2),mean,std,same speaker continues,speaker change,vs GRU + attention
0,GRU,0.0003,0.7775,0.7781,0.7781,0.7779,0.0004,0.7706,0.7811,0.0039
1,GRU + attention,0.0003,0.7743,0.7738,0.7740,0.7740,0.0003,0.7727,0.7738,0.0000
2,transformer,0.0003,0.7765,0.7761,0.7764,0.7763,0.0002,0.7841,0.7708,0.0023
3,transformer + ALiBi,0.0003,0.7744,0.7740,0.7736,0.7740,0.0004,0.7804,0.7693,-0.0000
4,transformer + speaker bias,0.0003,0.7763,0.7761,0.7763,0.7762,0.0001,0.7846,0.7704,0.0022
5,transformer + ALiBi + speaker bias,0.0003,0.7738,0.7740,0.7737,0.7738,0.0002,0.7827,0.7677,-0.0002


### Is the best transformer really better or worse than the recurrent networks?

Paired bootstrap over test conversations, as in the main notebook. For each model, the per-conversation errors are averaged over its 3 seeds first, so the comparison uses the seed-averaged model quality, and the 1,000 resamples cover the choice of test conversations. Negative = the transformer variant is better.

In [9]:
rng = np.random.default_rng(0)
n_test = len(splits["test"])
resamples = rng.integers(0, n_test, size=(1000, n_test))


def seed_average(name):
    sse = np.mean([e["all"][0] for e in errors[name]], axis=0)
    return sse, errors[name][0]["all"][1]


rows = []
for variant in [m for m in MODELS if m.startswith("transformer")]:
    sse_t, count = seed_average(variant)
    for recurrent in ("GRU", "GRU + attention"):
        sse_r, _ = seed_average(recurrent)
        difference = sse_t[resamples].sum(1) / count[resamples].sum(1) - sse_r[resamples].sum(1) / count[resamples].sum(1)
        low, high = np.percentile(difference, [2.5, 97.5])
        rows.append({"comparison": f"{variant} minus {recurrent}",
                     "difference": sse_t.sum() / count.sum() - sse_r.sum() / count.sum(),
                     "95% CI low": low, "95% CI high": high})
display(pd.DataFrame(rows).round(4))

,comparison,difference,95% CI low,95% CI high
0,transformer minus GRU,-0.0016,-0.0021,-0.0011
1,transformer minus GRU + attention,0.0023,0.0018,0.0027
2,transformer + ALiBi minus GRU,-0.0039,-0.0044,-0.0034
3,transformer + ALiBi minus GRU + attention,-0.0000,-0.0004,0.0004
4,transformer + speaker bias minus GRU,-0.0017,-0.0021,-0.0012
5,transformer + speaker bias minus GRU + attention,0.0022,0.0018,0.0027
6,transformer + ALiBi + speaker bias minus GRU,-0.0041,-0.0047,-0.0035
7,transformer + ALiBi + speaker bias minus GRU +...,-0.0002,-0.0007,0.0002


### What the speaker biases learned

The learned bias per layer and head, for the best seed-0 model with a speaker bias. Positive = attends more to that speaker's turns.

In [10]:
for name in [m for m in MODELS if "speaker" in m]:
    model = load_model(name, chosen_lr[name], SEEDS[0])
    b = model.speaker_bias.detach().cpu().numpy()
    table = pd.DataFrame({f"layer {l + 1}": [f"{b[l, h, 0]:+.2f} / {b[l, h, 1]:+.2f}" for h in range(b.shape[1])]
                          for l in range(b.shape[0])}, index=[f"head {h + 1}" for h in range(b.shape[1])])
    print(f"{name}: same speaker / other speaker")
    display(table)

transformer + speaker bias: same speaker / other speaker


,layer 1,layer 2,layer 3,layer 4
head 1,-0.11 / +0.11,-0.22 / +0.22,-0.07 / +0.07,-0.03 / +0.03
head 2,-0.13 / +0.13,-0.21 / +0.21,-0.05 / +0.05,-0.02 / +0.02
head 3,-0.05 / +0.05,-0.09 / +0.08,-0.12 / +0.12,-0.02 / +0.02
head 4,-0.03 / +0.03,-0.18 / +0.18,-0.04 / +0.04,+0.01 / -0.01


transformer + ALiBi + speaker bias: same speaker / other speaker


,layer 1,layer 2,layer 3,layer 4
head 1,-0.14 / +0.11,-0.14 / +0.14,-0.17 / +0.17,-0.17 / +0.16
head 2,-0.15 / +0.06,-0.19 / +0.18,-0.11 / +0.10,-0.10 / +0.08
head 3,+0.04 / -0.16,-0.07 / +0.05,-0.17 / +0.10,-0.13 / +0.13
head 4,-0.21 / +0.13,-0.15 / +0.15,-0.10 / +0.06,-0.08 / +0.08


### A simple baseline: each speaker's own average so far

Predict every feature of the next turn as the average of that **same speaker's** earlier turns in the conversation (the training mean if they haven't spoken yet). Nothing is learned. Unlike the models, this rule uses one extra piece of information: **who speaks next**, since it has to pick that person's history. The models predict turn t+1 without knowing whether the same person continues.

In [11]:
test_turns = turns[turns.conv_id.isin(test_ids)].sort_values(["conv_id", "turn_id"]).copy()
z = (test_turns[FEATURES].to_numpy(np.float32) - feature_mean) / feature_std
for j, feature in enumerate(FEATURES):
    test_turns[feature] = z[:, j]
own_average = (test_turns.groupby(["conv_id", "side"])[FEATURES]
               .transform(lambda s: s.shift().expanding().mean()).to_numpy())
own_average = np.where(np.isnan(own_average), 0.0, own_average)   # first turn by this speaker: training mean
previous_side = test_turns.groupby("conv_id").side.shift()
kind = np.where(previous_side.isna(), 0, np.where(previous_side == test_turns.side, 1, 2))
measured = ~np.isnan(z)


def baseline_mse(rows):
    mask = measured & rows[:, None]
    return float(np.mean((own_average[mask] - z[mask]) ** 2))


display(pd.DataFrame([{"model": "speaker's own average so far (knows who speaks next)",
                       "test MSE": baseline_mse(np.ones(len(z), bool)),
                       "same speaker continues": baseline_mse(kind == 1),
                       "speaker change": baseline_mse(kind == 2)}]).round(4))

,model,test MSE,same speaker continues,speaker change
0,speaker's own average so far (knows who speaks...,0.7056,0.6273,0.7483


## 8. What the results say

### A recency bias lets the transformer catch up with the RNN, but not overtake it

Test MSE, mean over 3 seeds (the spread across seeds is 0.0001 to 0.0004):

| model | test MSE | minus GRU + attention (95% CI) |
|---|---|---|
| GRU + attention | 0.7740 | |
| transformer + ALiBi + speaker bias | 0.7738 | −0.0002 (−0.0007 to +0.0002) |
| transformer + ALiBi | 0.7740 | −0.0000 (−0.0004 to +0.0004) |
| transformer + speaker bias | 0.7762 | +0.0022 (+0.0018 to +0.0027) |
| transformer | 0.7763 | +0.0023 (+0.0018 to +0.0027) |
| GRU | 0.7779 | +0.0039 |

- **The gap from the first notebook is real.** The plain transformer is 0.0023 behind GRU + attention, and its 3 seeds differ by only 0.0002.
- **ALiBi closes the whole gap.** With a recency bias, the transformer ties GRU + attention: the interval includes 0. It beats the plain GRU clearly (−0.0039, CI −0.0044 to −0.0034). So a transformer can *match* the best recurrent network here, but this experiment gives no evidence that it can *outperform* it.
- **ALiBi helps where expected, and a little more.** When the same speaker continues, the transformer's error drops from 0.784 to 0.780. At speaker changes it also improves (0.771 to 0.769). It is still worse than GRU + attention when the same speaker continues (0.780 vs 0.773) and better at speaker changes (0.769 vs 0.774): the two architectures make different kinds of errors.
- **The speaker bias adds nothing measurable.** Alone it changes test MSE by 0.0001, and on top of ALiBi by 0.0002, within the noise. The learned values are small (between −0.2 and +0.2) and mostly *favor the other speaker's turns*. The likely reason it doesn't help: each turn's speaker is already part of its input, so attention can already tell same-speaker turns from other-speaker turns.

### A simple baseline beats every model

Averaging the same speaker's earlier turns scores **0.706** on the test set, better than every model here (0.774). It is much better when the same speaker continues (0.627 against about 0.77 for the models) and still better at speaker changes (0.748 against 0.768 to 0.781).

The baseline's advantage is knowing **who speaks next**. The models never get that, so at every turn they have to hedge between the two speakers' typical voices. In a dialogue system the next speaker is known, and McNeill & Levitan (2023) give their decoder the role of the upcoming speaker. **Adding the next speaker as an input is the obvious next experiment.** It will likely help every model more than any architecture change tested here, and it may change which architecture wins.

### Limits

- One train/validation/test split. Seeds vary the initial weights, dropout and batch order.
- The transformer sizes come from the sweep in the first notebook. They were not re-tuned with ALiBi.


## 9. Exploratory: telling the models who speaks next

Section 8 showed that a rule which knows the next speaker beats every model. Here each model gets one extra input at every turn t: whether turn t+1 is the **same speaker continuing**, a **speaker change**, or the first turn. Together with the current speaker, already in the input, this says exactly who speaks next. Nothing about the speech of turn t+1 is revealed.

McNeill & Levitan (2023) give their decoder the role of the upcoming speaker in the same way. Each model uses learning rate 3e-4, which every model chose in section 6, and 3 seeds.

In [12]:
class NextSpeakerEncoder(nn.Module):
    # Same as TurnEncoder, plus a one-hot of the next turn's type: first turn / same speaker continues / speaker change.
    def __init__(self, model_dim=128):
        super().__init__()
        self.proj = nn.Linear(2 * N_FEAT + 2 + 3, model_dim)

    def forward(self, batch):
        next_type = nn.functional.one_hot(batch["turn_type"].clamp(min=0), 3).to(batch["inputs"].dtype)  # padding (-1) -> 0
        return self.proj(torch.cat([batch["inputs"], next_type], dim=-1))


NEXT_MODELS = {
    "GRU + next speaker": lambda: GRUModel(NextSpeakerEncoder(), hidden=256),
    "GRU + attention + next speaker": lambda: GRUModel(NextSpeakerEncoder(), hidden=128, attention=True),
    "transformer + next speaker": lambda: BiasedTransformer(NextSpeakerEncoder(256), alibi=False, speaker_bias=False),
    "transformer + ALiBi + next speaker": lambda: BiasedTransformer(NextSpeakerEncoder(256), alibi=True, speaker_bias=False),
}
MODELS.update(NEXT_MODELS)

for name in NEXT_MODELS:
    torch.manual_seed(0)
    model = MODELS[name]().eval()
    with torch.no_grad():
        original, changed = model(check_batch), model(perturb_after(check_batch, k))
    assert torch.allclose(original[:, :k + 1], changed[:, :k + 1], atol=1e-5), name
print("causality check passed for the next-speaker models")

NEXT_LR = 3e-4
for name in NEXT_MODELS:
    print(name, flush=True)
    for seed in SEEDS:
        run(name, NEXT_LR, seed)

causality check passed for the next-speaker models
GRU + next speaker


GRU + attention + next speaker


transformer + next speaker


transformer + ALiBi + next speaker


In [13]:
rows = []
for name in NEXT_MODELS:
    per_seed = []
    for seed in SEEDS:
        model = load_model(name, NEXT_LR, seed)
        per_seed.append({kind: per_conversation_errors(model, loaders["test"], turn_type)
                         for kind, turn_type in [("all", None), ("same speaker", 1), ("speaker change", 2)]})
    errors[name] = per_seed
    test = [e["all"][0].sum() / e["all"][1].sum() for e in per_seed]
    rows.append({"model": name, **{f"test MSE (seed {s})": t for s, t in zip(SEEDS, test)},
                 "mean": np.mean(test), "std": np.std(test, ddof=1),
                 "same speaker continues": np.mean([e["same speaker"][0].sum() / e["same speaker"][1].sum() for e in per_seed]),
                 "speaker change": np.mean([e["speaker change"][0].sum() / e["speaker change"][1].sum() for e in per_seed])})
rows.append({"model": "speaker's own average so far (section 8)", "mean": baseline_mse(np.ones(len(z), bool)),
             "same speaker continues": baseline_mse(kind == 1), "speaker change": baseline_mse(kind == 2)})
display(pd.DataFrame(rows).round(4))

rows = []
for variant in ("transformer + next speaker", "transformer + ALiBi + next speaker"):
    sse_t, count = seed_average(variant)
    for recurrent in ("GRU + next speaker", "GRU + attention + next speaker"):
        sse_r, _ = seed_average(recurrent)
        difference = sse_t[resamples].sum(1) / count[resamples].sum(1) - sse_r[resamples].sum(1) / count[resamples].sum(1)
        low, high = np.percentile(difference, [2.5, 97.5])
        rows.append({"comparison": f"{variant} minus {recurrent}", "difference": sse_t.sum() / count.sum() - sse_r.sum() / count.sum(),
                     "95% CI low": low, "95% CI high": high})
print("With the next speaker known (negative = the transformer is better):")
display(pd.DataFrame(rows).round(4))

,model,test MSE (seed 0),test MSE (seed 1),test MSE (seed 2),mean,std,same speaker continues,speaker change
0,GRU + next speaker,0.6556,0.6555,0.6569,0.6560,0.0007,0.5773,0.6993
1,GRU + attention + next speaker,0.6533,0.6536,0.6553,0.6541,0.0011,0.5745,0.6979
2,transformer + next speaker,0.6542,0.6535,0.6540,0.6539,0.0004,0.5752,0.6972
3,transformer + ALiBi + next speaker,0.6527,0.6524,0.6529,0.6526,0.0003,0.5741,0.6958
4,speaker's own average so far (section 8),NaN,NaN,NaN,0.7056,NaN,0.6273,0.7483


With the next speaker known (negative = the transformer is better):


,comparison,difference,95% CI low,95% CI high
0,transformer + next speaker minus GRU + next sp...,-0.0021,-0.0026,-0.0016
1,transformer + next speaker minus GRU + attenti...,-0.0002,-0.0006,0.0003
2,transformer + ALiBi + next speaker minus GRU +...,-0.0033,-0.0039,-0.0029
3,transformer + ALiBi + next speaker minus GRU +...,-0.0014,-0.0019,-0.0010


### What it shows

**Knowing who speaks next is the biggest improvement in this project, and with it the transformer pulls ahead.** Test MSE, mean of 3 seeds:

| model | with the next speaker | without (section 7) |
|---|---|---|
| transformer + ALiBi | **0.6526** | 0.7740 |
| transformer | 0.6539 | 0.7763 |
| GRU + attention | 0.6541 | 0.7740 |
| GRU | 0.6560 | 0.7779 |
| speaker's own average so far | 0.7056 | 0.7056 |

- Every model's error drops by about 15%, far more than any architecture change. Every model now beats the simple speaker-average rule comfortably.
- **Transformer + ALiBi minus GRU + attention: −0.0014 (95% CI −0.0019 to −0.0010).** This is the first setup where the transformer *outperforms* the best recurrent network. The seeds differ by at most 0.0011, so the gap is real, although small (about 0.2% of the error).
- The transformer gains more from this input than the GRUs do. Even without ALiBi it moves from 0.0023 behind GRU + attention to level with it. That fits section 8, where it was already the best model at speaker changes. Once it knows whose turn is next, attention can look up that person's earlier turns directly.
- ALiBi still adds a small, real gain on top (0.6539 to 0.6526).

## 10. Does regularization let the larger transformer pull ahead?

The transformer has about 18 times more parameters than GRU + attention but only ties it. A larger model often needs more regularization before its extra capacity helps, so both models get the same small grid, on the main experiment (speech features only, no next speaker):

- **Dropout** 0.1, 0.2 or 0.3.
- **Weight decay** 0.01 or 0.1, with AdamW (decoupled weight decay, as in McNeill & Levitan 2023). The earlier runs used plain Adam without weight decay.
- Learning rate 3e-4, which every model chose in section 6.

Each grid point is trained with seed 0 and compared on validation. The best point per model is then trained with seeds 1 and 2, and its test MSE is compared with the unregularized model from section 7.

In [14]:
DROPOUTS, WEIGHT_DECAYS, REG_LR = [0.1, 0.2, 0.3], [0.01, 0.1], 3e-4
REG_BASE = {
    "GRU + attention": lambda dropout: GRUModel(TurnEncoder(), hidden=128, attention=True, dropout=dropout),
    "transformer + ALiBi": lambda dropout: BiasedTransformer(TurnEncoder(256), alibi=True, speaker_bias=False, dropout=dropout),
}
for base, build in REG_BASE.items():
    for dropout in DROPOUTS:
        MODELS[f"{base} (dropout {dropout:g})"] = (lambda build=build, dropout=dropout: build(dropout))

reg_rows, reg_best = [], {}
for base in REG_BASE:
    print(base, flush=True)
    for dropout in DROPOUTS:
        for weight_decay in WEIGHT_DECAYS:
            name = f"{base} (dropout {dropout:g})"
            record = run(name, REG_LR, SEEDS[0], weight_decay)
            reg_rows.append({"model": base, "dropout": dropout, "weight decay": weight_decay,
                             "best epoch": record["best epoch"], "val MSE (seed 0)": record["val MSE"]})
            if base not in reg_best or record["val MSE"] < reg_best[base][0]:
                reg_best[base] = (record["val MSE"], name, weight_decay)
    _, name, weight_decay = reg_best[base]
    for seed in SEEDS[1:]:
        run(name, REG_LR, seed, weight_decay)
print("Every grid point (seed 0, validation):")
display(pd.DataFrame(reg_rows).round(4))

GRU + attention


transformer + ALiBi


  -> transformer + ALiBi (dropout 0.1) | lr 0.0003 | wd 0.01 | seed 0: val MSE 0.7607, best epoch 25, 4.0 min


  -> transformer + ALiBi (dropout 0.1) | lr 0.0003 | wd 0.1 | seed 0: val MSE 0.7607, best epoch 20, 4.4 min


  -> transformer + ALiBi (dropout 0.2) | lr 0.0003 | wd 0.01 | seed 0: val MSE 0.7612, best epoch 25, 6.4 min


  -> transformer + ALiBi (dropout 0.2) | lr 0.0003 | wd 0.1 | seed 0: val MSE 0.7620, best epoch 21, 5.0 min


... (2 outputs left out)

  -> transformer + ALiBi (dropout 0.1) | lr 0.0003 | wd 0.1 | seed 1: val MSE 0.7605, best epoch 23, 5.8 min


  -> transformer + ALiBi (dropout 0.1) | lr 0.0003 | wd 0.1 | seed 2: val MSE 0.7601, best epoch 21, 5.8 min


Every grid point (seed 0, validation):


,model,dropout,weight decay,best epoch,val MSE (seed 0)
0,GRU + attention,0.1,0.01,32,0.7610
1,GRU + attention,0.1,0.10,32,0.7610
2,GRU + attention,0.2,0.01,32,0.7614
3,GRU + attention,0.2,0.10,32,0.7614
4,GRU + attention,0.3,0.01,32,0.7619
5,GRU + attention,0.3,0.10,36,0.7616
6,transformer + ALiBi,0.1,0.01,25,0.7607
7,transformer + ALiBi,0.1,0.10,20,0.7607
8,transformer + ALiBi,0.2,0.01,25,0.7612
9,transformer + ALiBi,0.2,0.10,21,0.7620


In [15]:
rows = []
for base, (_, name, weight_decay) in reg_best.items():
    per_seed = [per_conversation_errors(load_model(name, REG_LR, seed, weight_decay), loaders["test"]) for seed in SEEDS]
    errors[f"{base}, regularized"] = [{"all": e} for e in per_seed]
    test = [sse.sum() / count.sum() for sse, count in per_seed]
    plain = results.set_index("model").loc[base, "mean"]
    rows.append({"model": base, "kept": f"{name.split('(')[1][:-1]}, weight decay {weight_decay:g}",
                 **{f"test MSE (seed {s})": t for s, t in zip(SEEDS, test)}, "mean": np.mean(test), "std": np.std(test, ddof=1),
                 "unregularized (section 7)": plain, "change": np.mean(test) - plain})
display(pd.DataFrame(rows).round(4))

sse_t, count = seed_average("transformer + ALiBi, regularized")
sse_r, _ = seed_average("GRU + attention, regularized")
difference = sse_t[resamples].sum(1) / count[resamples].sum(1) - sse_r[resamples].sum(1) / count[resamples].sum(1)
low, high = np.percentile(difference, [2.5, 97.5])
print(f"regularized transformer + ALiBi minus regularized GRU + attention: {sse_t.sum() / count.sum() - sse_r.sum() / count.sum():+.4f} "
      f"(95% CI {low:+.4f} to {high:+.4f}; negative = the transformer is better)")

,model,kept,test MSE (seed 0),test MSE (seed 1),test MSE (seed 2),mean,std,unregularized (section 7),change
0,GRU + attention,"dropout 0.1, weight decay 0.1",0.7742,0.7737,0.7737,0.7739,0.0003,0.774,-0.0001
1,transformer + ALiBi,"dropout 0.1, weight decay 0.1",0.7739,0.7734,0.7737,0.7737,0.0003,0.774,-0.0003


regularized transformer + ALiBi minus regularized GRU + attention: -0.0002 (95% CI -0.0006 to +0.0002; negative = the transformer is better)


### What it shows

**Regularization doesn't change the picture.**

- **GRU + attention:** every grid point lands between 0.7610 and 0.7619 on validation. Weight decay changes nothing, and more dropout makes it slightly worse.
- **Transformer + ALiBi:** dropout 0.1 is best (0.7607 with either weight decay). Dropout 0.2 is slightly worse, and dropout 0.3 is clearly too much (0.7654 and 0.7723, stopping after 6 to 14 epochs).
- **On the test set, the kept settings barely move:** 0.7739 for GRU + attention (−0.0001 against section 7) and 0.7737 for the transformer (−0.0003).
- **Regularized transformer minus regularized GRU + attention: −0.0002 (95% CI −0.0006 to +0.0002):** still a tie.

So the transformer's extra parameters aren't being held back by overfitting. Without the next speaker, it is limited by what its inputs can tell it, not by regularization.